In [ ]:
sample_count = 100
trial = 1

In [ ]:
import re
import json
import pandas as pd
from bs4 import BeautifulSoup
from tqdm import tqdm
tqdm.pandas()

## Llama 7B

In [ ]:
from langchain.chains import LLMChain

# llm
from langchain.callbacks.manager import CallbackManager
from langchain.callbacks.streaming_stdout import StreamingStdOutCallbackHandler
from langchain_community.llms import LlamaCpp

# Prompt
from langchain.chains.prompt_selector import ConditionalPromptSelector
from langchain.prompts import PromptTemplate

# Parser
from langchain_core.output_parsers import StrOutputParser

In [ ]:
# TODO: Make it give a formatted response so NER isn't necesssary.

# Prompt for LLM to do its geolocation task
prompt = PromptTemplate(
    input_variables=["headline", "body"],
    template="""<<SYS>> \n You are an assistant tasked in geo-locating this news article. \n <</SYS>> \n\n 
    [INST] Generate a SHORT response of where you think this article is talking about. BE SPECIFIC AS POSSIBLE. IT IS IMPERATIVE THAT YOU HIGHLIGHT THE MOST SPECIFIC LOCATION. Give your response in the following format: \
    1.If there is one, the city the article is talking about. Otherwise, state that it can't be located. \n 
    2.The specific location within the city you got if you found one. \
    3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision. \
    If you do not know, PLEASE GIVE THE BEST GUESS AS POSSIBLE. This is the article: \n\n
    Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n [/INST]""",
)


In [ ]:
prompt3_1 = PromptTemplate(
    input_variables=["headline", "body"],
    template="""
    <|begin_of_text|><|start_header_id|>system<|end_header_id|>
    
    Cutting Knowledge Date: December 2023
    Today Date: 26 Jul 2024

    You are an expert in geo-location and have a deep understanding of specific places and organizations. In a short response, your task is to identify and provide the most exact real location mentioned in the news article. This can be a place, organization, facility, or any location that can help identify where the article takes place or talks about. Also, mention any specific locations or organizations explicitly found within the article that influenced your decision. If you cannot determine a location, state that explicitly. Do not discuss anthing else.<|eot_id|><|start_header_id|>user<|end_header_id|>

    You are tasked in geo-locating this news article. Generate a SHORT response specifying the most exact location you can find mentioned in the article. Give your answer in the following format:
    1. If there is one, the city the article is talking about. Otherwise, state that it can't be located. 
    2. The specific place within the city you got if you found one.
    3. The involved specific locations or organizations EXPLICITLY FOUND WITHIN THE ARTICLE that influenced your decision.
     
    If you cannot determine a location, state that explicitly. DO NOT MAKE UP INFORMATION. This is the news article:
    Headline: \n\n {headline} \n\n Body: \n\n {body} \n\n <|eot_id|><|start_header_id|>assistant<|end_header_id|>""",
)


In [ ]:
# Call model. Needs to be downloaded
llama_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"
llama2_model_path = "./models/llama_7B/llama-2-7b-chat.Q4_K_M.gguf"
llama3_1_model_path = "./models/llama_3_1_8B/Meta-Llama-3.1-8B-Instruct-Q4_K_M.gguf"


In [ ]:
llm2 = LlamaCpp(
    model_path=llama2_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
)
output_parser = StrOutputParser()

In [ ]:
llm3_1 = LlamaCpp(
    model_path=llama3_1_model_path,
    n_gpu_layers=1,
    n_batch=1024,
    n_ctx=2048,
    f16_kv=True,
)
output_parser = StrOutputParser()

In [ ]:
chain2 = prompt | llm2 | output_parser
chain3_1 = prompt3_1 | llm3_1 | output_parser

In [ ]:
# Run LLM on a given article
def run_llm2(headline, body):
    return chain2.invoke({"headline": headline, "body": body})

def run_llm3_1(headline, body):
    return chain3_1.invoke({"headline": headline, "body": body})


## NER Model

In [ ]:
import spacy
from span_marker import SpanMarkerModel

In [ ]:
# Load the spacy model with the span_marker pipeline component
nlp = spacy.load("en_core_web_sm", exclude=["ner"])
nlp.add_pipe("span_marker", config={"model": "tomaarsen/span-marker-roberta-large-ontonotes5"})

## Google Maps

In [ ]:
# Load environment variables
import os
from dotenv import load_dotenv

load_dotenv()
gmap_api_key =  os.getenv('GMAP_API_KEY')

In [ ]:
import ast
import requests
import googlemaps
from mapbox import Geocoder

In [ ]:
gmap_client_key = gmap_api_key
gmaps = googlemaps.Client(key=gmap_client_key)

In [ ]:
# Google Maps API handler
def callGoogleMapsAPI(location):
    try:
        # If it's in boston, we can do a more specific search
        if ("Boston" in location):
            location = f"{location}, Boston"
            
        # Locations are limited to Massachusetts for now
        geocode_result = gmaps.geocode(f"{location}, Massachussetts", components={"administrative_area_level": "MA", "country": "US"})
        
        if (len(geocode_result) > 0):
            longitude = geocode_result[0]['geometry']['location']['lng']
            latitude = geocode_result[0]['geometry']['location']['lat']
            return longitude, latitude
        else:
            return None
    except Exception as error:
        print(error)
        return None

### Functions to manage caches

In [ ]:
# Load the cache from the file at the start
def load_cache(path):
    try:
        with open(path, 'r') as file:
            cache = json.load(file)
    except FileNotFoundError:
        cache = {}
    return cache

# Save cache to file
def save_cache_to_file(cache, path):
    with open(path, 'w') as file:
        json.dump(cache, file, indent=4)

### Wrapper function to measure time taken by a given function

In [ ]:
import time

def sec_to_hms(seconds):
    hours = int(seconds // 3600)
    minutes = int((seconds % 3600) // 60)
    remaining_seconds = round(seconds % 60)
    return f"{hours:02}:{minutes:02}:{remaining_seconds:02}"

def check_time(func):
    
    def wrapper(*args, **kwargs):
        start_time = time.time()
        result = func(*args, **kwargs)
        end_time = time.time()
        total_time = end_time - start_time
        total_time_formatted = sec_to_hms(total_time)
        print(f"Time taken: {total_time_formatted}")
        return result
    return wrapper

## Pipeline Entry Point

In [ ]:
sample_data_path = "./sample_data/Articles_Nov_2020_March_2023.csv" # Using this as I don't have the other one

In [ ]:
# Temporary. Use given article data set. Comment out when obtain the other data sate
full_df = pd.read_csv(sample_data_path)

# Format data set to match expected pipeline input
full_df = full_df.rename(columns={"Headline": "hl1", "Body": "body"})

# Make 'tagging' column be the id column
tagging_col = full_df.pop('Tagging')
full_df.insert(0, '_id', tagging_col)

# Drop rows where at least one of the specified columns is empty
columns_to_check = ['_id', 'hl1', 'body'] 
full_df = full_df.dropna(subset=columns_to_check, how='all')

# Drop empty rows too
full_df = full_df[~full_df['body'].apply(lambda x: isinstance(x, float))]
full_df = full_df[~full_df['hl1'].apply(lambda x: isinstance(x, float))]

# Pick a random sample of 5 articles
raw_df = full_df.sample(sample_count)
# raw_df = full_df
len(raw_df)


In [ ]:
raw_df.head(10)

The ML Model honestly just needs the `id`, `header`, and `body`.

In [ ]:
df = pd.concat([raw_df['_id'], raw_df['hl1'], raw_df['body']], axis=1)

Remove Duplicates (if any)

In [ ]:
duplicates = df.duplicated(subset=['hl1'])

In [ ]:
print(duplicates.value_counts())

In [ ]:
df = df.drop_duplicates(subset=['hl1'])

Clean the HTML in the body and header

In [ ]:
# Function to extract the text from the html of the article
func_clean_html = lambda text: BeautifulSoup(text, "html.parser").get_text()
df['body'] = df['body'].progress_apply(func_clean_html)
df['hl1'] = df['hl1'].progress_apply(func_clean_html)

Clean the Body and Header with Regex

In [ ]:
# Function to remove extra symbols from the text
func_clean_regex = lambda text: ' '.join([word for word in re.findall(r'[A-Za-z0-9!@#$%^&*().]+', text) if len(word) > 1])
df['body'] = df['body'].progress_apply(func_clean_regex)
df['hl1'] = df['hl1'].progress_apply(func_clean_regex)

### Benchmark total times

In [ ]:
time_df = pd.DataFrame(columns=['NER', 'LLM2', 'LLM3.1'])

### Explicit Article Mentions

Load the well-known locations, organizations, and neighborhoods dictionary and the locations that we don't want to allow (i.e. Too broad or incorrect ones like "Boston", "Massachussets", etc.)

In [ ]:
known_title_locs_path = "./geodata/known_locs.json"
known_title_locs = load_cache(known_title_locs_path)

unwanted_entities_path = "./geodata/unwanted_locations.json"  
unwanted_entities = load_cache(unwanted_entities_path)

In [ ]:
# If a location is in the title, use that as the article's location
def explicit_filtering(header):
    # Look through the header for known locations
    locations_list = []
    lowercase_header = header.lower()
    for location in known_title_locs.keys():
        if (location.lower() in lowercase_header):
            if location not in unwanted_entities["FAC"]:
                locations_list.append(location)
    
    print("\n Locations found on title: \n {locations_list} \n")
    return locations_list

In [ ]:
# df["Explicit_Pass"] = df["hl1"].progress_apply(explicit_filtering)
df["Explicit_Pass"] = None


In [ ]:
df["Explicit_Pass"].value_counts().head(10)

### NER Code First Pass

In [ ]:
# Return all valid facilities and organizations found
def get_valid_entities(entities):
    valid_facilities = []
    valid_orgs = []

    for entity in entities:
        if (entity.label_ == "FAC" and entity.text not in unwanted_entities["FAC"]):
            valid_facilities.append(entity.text)
        
        if (entity.label_ == "ORG" and entity.text not in unwanted_entities["ORG"]):
            valid_orgs.append(entity.text)
    
    valid_entities = valid_facilities + valid_orgs

    if (len(valid_entities) == 0):
        return None
    else:
        return valid_entities
        

In [ ]:
# Run NER on the body of the article and return first valid facility
def run_NER(text):
    try:
        if (text == None or text == ""):
            return None
        
        entities = nlp(text).ents
        valid_entities = get_valid_entities(entities)
        return valid_entities
        
    except Exception as error:
        print(error)
        return None

In [ ]:
# Chunk processing - split the article into chunks of text and run NER on each chunk
chunk_size = 100
def chunk_processing(text, chunk_size=chunk_size, chunk_limit=None):
    # Split text into smaller chunks
    chunks = split_text_into_chunks(text, chunk_size)

    all_entities = []

    if chunk_limit is not None:
        chunks = chunks[:chunk_limit]
    
    # Process each chunk and return if valid entities are found
    for chunk in chunks:
        result = run_NER(chunk)
        if result is not None:
             all_entities.extend(result)
    
    print("\n All entities for the article: \n {all_entities} \n")
    
    if len(all_entities) == 0:
        return None
    else:
        return all_entities

# Split article text into chunks of specified size
def split_text_into_chunks(text, chunk_size=chunk_size):
    words = text.split()
    chunks = [' '.join(words[i:i + chunk_size]) for i in range(0, len(words), chunk_size)]
    return chunks

In [ ]:
@check_time
def handle_chunk_processing(article):
    # Filter out articles that have no text
    text = article['body']
    if (text is None or text == ""):
            return None
    try:
        return chunk_processing(text)
    except Exception as error:
        print(error)

In [ ]:
start_time = time.time()

df["NER_Pass"] = df.progress_apply(handle_chunk_processing, axis=1)
# df["NER_Pass"] = None

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Time taken: {total_time_formatted}")
time_df.loc[0, "NER"] = total_time_formatted


In [ ]:
df.head(10)

### Llama Prediction

In [ ]:
def filter_llama_output(log):
    # Define regex patterns to match the lines we want to remove
    llama_print_timings_pattern = re.compile(r'llama_print_timings:.*')
    llama_generate_pattern = re.compile(r'Llama.generate:.*')

    lines = log.split('\n')

    filtered_lines = []

    for line in lines:
        # If the line matches any of the unwanted patterns, skip it
        if llama_print_timings_pattern.match(line) or llama_generate_pattern.match(line):
            continue

        filtered_lines.append(line.strip())

    # Join the filtered lines back into a single string
    filtered_log = '\n'.join(filtered_lines)
    
    return filtered_log

In [ ]:
# Run LLM model on the articles, and then run the NER on the prediction.
@check_time
def predict_llama2(article):
    try:
        truncated_text = article['body'][:6000]
        llama_prediction = run_llm2(article['hl1'], truncated_text)
        cleaned_prediction = filter_llama_output(llama_prediction)
        print("\n Llama 2 Prediction: \n {cleaned_prediction} \n")
        
        valid_entities = run_NER(cleaned_prediction)
        print("\n All entities for the article from LLM 2: \n {valid_entities} \n")
        return valid_entities
    
    except Exception as error:
        print(error)
        return None

In [ ]:
start_time = time.time()

df['LLM_2_Pass'] = df.progress_apply(predict_llama2, axis=1)
# df['LLM_2_Pass'] = None

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Total time taken: {total_time_formatted}")
time_df.loc[0, "LLM2"] = total_time_formatted

In [ ]:
@check_time
def predict_llama3_1(article):
    try:
        truncated_text = article['body'][:6000]
        llama_prediction = run_llm3_1(article['hl1'], truncated_text)
        cleaned_prediction = filter_llama_output(llama_prediction)
        print("\n Llama 3.1 Prediction: \n {cleaned_prediction} \n")

        valid_entities = run_NER(cleaned_prediction)
        print("\n All entities for the article from LLM 3.1: \n {valid_entities} \n")
        return valid_entities
    
    except Exception as error:
        print(error)
        return None

In [ ]:
start_time = time.time()

df['LLM_3_1_Pass'] = df.progress_apply(predict_llama3_1, axis=1)

end_time = time.time()
total_time = end_time - start_time
total_time_formatted = sec_to_hms(total_time)
print(f"Total time taken: {total_time_formatted}")
time_df.loc[0, "LLM3.1"] = total_time_formatted

In [ ]:
df.head(10)

In [ ]:
df

In [ ]:
time_df

In [ ]:
print(df['Explicit_Pass'].value_counts().sum())
df['Explicit_Pass'].value_counts()

In [ ]:
print(df['NER_Pass'].value_counts().sum())
df['NER_Pass'].value_counts()

In [ ]:
print(df['LLM_2_Pass'].value_counts().sum())
df['LLM_2_Pass'].value_counts()

In [ ]:
print(df['LLM_3_1_Pass'].value_counts().sum())
df['LLM_3_1_Pass'].value_counts()

In [ ]:
len(df)

In [ ]:
df.to_csv(f"./results/benchmarking_{sample_count}_samples_{trial}_trial.csv")
time_df.to_csv(f"./results/benchmarking_times_{sample_count}_samples_{trial}_trial.csv")

In [ ]:
print(df.count())
print(df.dropna(subset=['NER_Pass'])["LLM_2_Pass"].notnull().sum())
print(df.dropna(subset=['NER_Pass'])["LLM_3_1_Pass"].notnull().sum())

print(df.dropna(subset=['LLM_2_Pass'])["LLM_3_1_Pass"].notnull().sum())